# 02 · BTC Price & Perpetual Swap Derivatives

**Purpose:** Model BTC spot price dynamics and the perpetual futures market — the most liquid BTC derivatives by volume.

Perpetual swaps are unique derivatives with no expiry that track spot via a funding rate mechanism:
- **Positive funding rate**: longs pay shorts → market is in contango (bullish positioning)
- **Negative funding rate**: shorts pay longs → market is in backwardation (bearish/hedged)

**Data sources:**
- Binance: `/fapi/v1/fundingRate`, `/futures/data/openInterestHist`, `/api/v3/klines`
- yfinance: `BTC-USD` OHLCV, `BTC=F` CME futures

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.binance import (
    get_spot_klines, get_funding_rate_history, get_funding_rate_daily,
    get_open_interest_history, get_long_short_ratio, get_spot_price
)
from src.data.yfinance_fetcher import fetch_ohlcv, fetch_prices
from src.utils.plotting import (
    plot_price_with_volume, plot_funding_rates, PLOTLY_TEMPLATE, BTC_ORANGE
)
from config import DEFAULT_LOOKBACK_DAYS

pd.options.display.float_format = '{:,.4f}'.format
print('Setup complete.')

## 1. BTC Spot Price

In [ ]:
# Fetch BTC/USD OHLCV from yfinance (reliable, cached)
btc_ohlcv = fetch_ohlcv('BTC-USD', period='2y')
btc_price = btc_ohlcv['Close']

print(f'BTC price range: ${btc_price.min():,.0f} – ${btc_price.max():,.0f}')
print(f'Latest close:    ${btc_price.iloc[-1]:,.0f}')
print(f'YTD return:      {(btc_price.iloc[-1]/btc_price[btc_price.index >= f"{btc_price.index[-1].year}-01-01"].iloc[0] - 1)*100:.1f}%')

In [ ]:
fig = plot_price_with_volume(btc_ohlcv, title='BTC/USD — Daily OHLCV')
fig.show()

## 2. Realized Volatility

In [ ]:
log_returns = np.log(btc_price / btc_price.shift(1)).dropna()

# Annualized realized vol at different windows
rv_7d   = log_returns.rolling(7).std()   * np.sqrt(365) * 100
rv_30d  = log_returns.rolling(30).std()  * np.sqrt(365) * 100
rv_90d  = log_returns.rolling(90).std()  * np.sqrt(365) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=rv_7d.index,  y=rv_7d,  name='7d RV',  line=dict(width=1.5, color='#F39C12')))
fig.add_trace(go.Scatter(x=rv_30d.index, y=rv_30d, name='30d RV', line=dict(width=2,   color=BTC_ORANGE)))
fig.add_trace(go.Scatter(x=rv_90d.index, y=rv_90d, name='90d RV', line=dict(width=2,   color='#E74C3C', dash='dot')))

fig.update_layout(
    title='BTC Realized Volatility (Annualized %)',
    yaxis_title='Annualized Vol (%)', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=400, hovermode='x unified',
)
fig.show()

print(f'Current 30d RV: {rv_30d.iloc[-1]:.1f}%')
print(f'Current 90d RV: {rv_90d.iloc[-1]:.1f}%')

## 3. CME Bitcoin Futures vs Spot Basis

In [ ]:
# BTC=F is the front-month CME Bitcoin futures contract on Yahoo Finance
try:
    cme_prices = fetch_prices(['BTC=F', 'BTC-USD'], period='1y')
    if 'BTC=F' in cme_prices.columns and 'BTC-USD' in cme_prices.columns:
        basis = cme_prices['BTC=F'] - cme_prices['BTC-USD']
        basis_pct = (basis / cme_prices['BTC-USD']) * 100

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=['CME Futures vs Spot', 'Basis (Futures − Spot) %'])

        fig.add_trace(go.Scatter(x=cme_prices.index, y=cme_prices['BTC-USD'],
                                  name='Spot', line=dict(color=BTC_ORANGE, width=2)), row=1, col=1)
        fig.add_trace(go.Scatter(x=cme_prices.index, y=cme_prices['BTC=F'],
                                  name='CME Front Month', line=dict(color='#3498DB', width=2, dash='dot')), row=1, col=1)

        colors = [BTC_ORANGE if v >= 0 else '#E74C3C' for v in basis_pct.fillna(0)]
        fig.add_trace(go.Bar(x=basis_pct.index, y=basis_pct, name='Basis %',
                              marker_color=colors), row=2, col=1)
        fig.add_hline(y=0, line_color='white', line_width=0.5, row=2, col=1)

        fig.update_layout(title='CME Bitcoin Futures Basis', template=PLOTLY_TEMPLATE, height=550)
        fig.show()
    else:
        print('CME futures data not available (BTC=F may require a Yahoo Finance subscription).')
except Exception as e:
    print(f'CME data fetch failed: {e}')

## 4. Perpetual Swap Funding Rates (Binance)

In [ ]:
# 8-hour funding rate — 3 payments per day
funding_8h = get_funding_rate_history(symbol='BTCUSDT', days=365)
funding_daily = get_funding_rate_daily(symbol='BTCUSDT', days=365)

print(f'Fetched {len(funding_8h)} funding rate observations')
print(f'\nFunding rate statistics (annualized):')  
print(funding_8h['annualized_rate'].describe().apply(lambda x: f'{x*100:.2f}%'))

In [ ]:
fig = plot_funding_rates(
    funding_daily.resample('W').mean(),  # weekly average for clarity
    rate_col='annualized_rate',
    title='BTC Perpetual Funding Rate — Weekly Average (Annualized)',
)
fig.show()

In [ ]:
# Cumulative cost of a long perpetual position
cumulative = funding_daily['cumulative_funding'] * 100  # as %

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cumulative.index, y=cumulative.values,
    mode='lines', fill='tozeroy',
    line=dict(color=BTC_ORANGE, width=2),
    fillcolor='rgba(247,147,26,0.15)',
    name='Cumulative Funding Paid by Longs',
))
fig.add_hline(y=0, line_color='white', line_width=0.5)
fig.update_layout(
    title='Cumulative Funding Cost of Long BTC Perpetual (1 year)',
    yaxis_title='Cumulative Funding (%)', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=400,
)
fig.show()

total_cost = cumulative.iloc[-1]
print(f'Total funding cost for long over 1 year: {total_cost:.2f}%')
print(f'Annualized cost: {total_cost:.2f}% per year for a long-only perp strategy')

## 5. Open Interest

In [ ]:
oi = get_open_interest_history(symbol='BTCUSDT', period='1d', days=365)
print(f'Latest OI: {oi["oi_btc"].iloc[-1]:,.0f} BTC  (${oi["oi_usd"].iloc[-1]/1e9:.1f}B USD)')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Open Interest (BTC)', 'BTC Price (for reference)'])

fig.add_trace(go.Scatter(
    x=oi.index, y=oi['oi_btc'],
    name='OI (BTC)', fill='tozeroy', line=dict(color='#3498DB', width=2),
    fillcolor='rgba(52,152,219,0.15)',
), row=1, col=1)

# align BTC price to OI index
btc_aligned = btc_price.reindex(oi.index, method='ffill')
fig.add_trace(go.Scatter(
    x=btc_aligned.index, y=btc_aligned.values,
    name='BTC Price', line=dict(color=BTC_ORANGE, width=2),
), row=2, col=1)

fig.update_layout(title='Binance BTC-USDT Perp Open Interest', template=PLOTLY_TEMPLATE, height=550)
fig.show()

## 6. Funding Rate vs Price Correlation

High positive funding rate + rising price → crowded long positioning → potential reversion signal.

In [ ]:
# Merge price and funding for correlation analysis
combined = pd.DataFrame({
    'btc_price': btc_price,
    'funding_annualized': funding_daily['annualized_rate'],
}).dropna()

combined['btc_fwd_30d_return'] = combined['btc_price'].pct_change(30).shift(-30) * 100

fig = go.Figure(go.Scatter(
    x=combined['funding_annualized'] * 100,
    y=combined['btc_fwd_30d_return'],
    mode='markers',
    marker=dict(color=BTC_ORANGE, size=5, opacity=0.6),
    text=combined.index.strftime('%Y-%m-%d'),
))
fig.update_layout(
    title='Funding Rate vs. Forward 30-day BTC Return',
    xaxis_title='Daily Funding Rate (Annualized %)',
    yaxis_title='BTC Return 30 days forward (%)',
    template=PLOTLY_TEMPLATE, height=450,
)

# Add trendline
import numpy.polynomial.polynomial as poly
valid = combined[['funding_annualized', 'btc_fwd_30d_return']].dropna()
coefs = poly.polyfit(valid['funding_annualized'], valid['btc_fwd_30d_return'], deg=1)
x_line = np.linspace(valid['funding_annualized'].min(), valid['funding_annualized'].max(), 50)
y_line = poly.polyval(x_line, coefs)
fig.add_trace(go.Scatter(x=x_line*100, y=y_line, mode='lines',
                          line=dict(color='red', width=2, dash='dot'), name='Trend'))
fig.show()

corr = valid.corr().loc['funding_annualized', 'btc_fwd_30d_return']
print(f'Correlation (funding rate → 30d forward return): {corr:.3f}')

## Summary

- BTC perpetual funding rates reflect market sentiment: sustained positive rates indicate bullish leverage positioning
- The cumulative funding cost quantifies the carry cost of long perpetual exposure vs spot
- Open interest growth alongside price signals leveraged bull runs; OI drops with price signal deleveraging events
- CME futures basis provides a regulated-market view of BTC forward pricing

**Next:** [03 · BTC Options & IV Surface →](./03_btc_options_iv_surface.ipynb)